# 配套实践 11-02：比较动作均值与多峰动作分布

本练习构造两组同样成功的 Action Chunk：一组从障碍上方绕行，另一组从下方绕行。我们分别训练单一 MSE 轨迹和双分量混合密度模型，观察“平均误差较小”为什么不等于动作安全。依赖：PyTorch、NumPy、Matplotlib；CPU 即可运行。

<a href="https://qi-robotics.github.io/robot-world-model-tutorial/intermediate/11-demonstrations-to-action-sequences/" target="_blank">在新标签页返回课程正文</a>

In [ ]:
import numpy as np  # 计算连续密度常数并整理可视化数据
import torch  # 构造多峰动作数据并优化两种模型
import matplotlib.pyplot as plt  # 绘制示范、损失和学习到的动作分布
from matplotlib.patches import Rectangle  # 在轨迹图中绘制中央障碍
torch.set_num_threads(2)  # 限制轻量实验使用的 CPU 线程
torch.manual_seed(112)  # 固定示范噪声、初始化与采样结果
np.random.seed(112)  # 固定 NumPy 侧随机过程
plt.rcParams["figure.dpi"] = 120  # 提高笔记本图像显示清晰度

## 1. 一个 Context 对应两种成功动作

每条动作块包含 20 个二维位置，从左侧起点移动到右侧目标。上绕与下绕的选择在当前简化 Context 中没有被区分，因此训练数据的条件分布天然具有两个模式。

In [ ]:
sample_count = 600  # 设置示范动作块数量
chunk_length = 20  # 设置每个 Action Chunk 的二维位置数量
phase = torch.linspace(0.0, 1.0, chunk_length)  # 建立从起点到目标的归一化时间
x_positions = phase * 2.0 - 1.0  # 让所有示范从横坐标负一移动到正一
mode_signs = torch.where(torch.rand(sample_count) > 0.5, 1.0, -1.0)  # 为每条示范随机选择上绕或下绕模式
y_positions = mode_signs[:, None] * 0.75 * torch.sin(torch.pi * phase)[None, :]  # 使用正负半正弦生成两条绕障路径
y_positions = y_positions + 0.03 * torch.randn(sample_count, chunk_length)  # 加入少量示范者差异和测量噪声
demonstrations = torch.stack([x_positions.repeat(sample_count, 1), y_positions], dim=-1)  # 组成样本乘时间乘二维动作张量
fig, axis = plt.subplots(figsize=(8.8, 4.8))  # 创建多峰示范轨迹图
for sample_index in range(80):  # 只绘制部分示范避免图像过密
    trajectory = demonstrations[sample_index].numpy()  # 取出一条二维 Action Chunk
    color = "#2563eb" if mode_signs[sample_index] > 0 else "#7c3aed"  # 为上绕和下绕分配不同颜色
    axis.plot(trajectory[:, 0], trajectory[:, 1], color=color, alpha=0.16)  # 以半透明曲线显示示范变化
axis.add_patch(Rectangle((-0.25, -0.25), 0.5, 0.5, facecolor="#cbd5e1", edgecolor="#475569", label="Obstacle"))  # 绘制必须绕开的中央障碍
axis.scatter([-1.0, 1.0], [0.0, 0.0], color="#16a34a", s=70, zorder=4, label="Start / goal")  # 标出共同起点和目标
axis.set(title="Two valid modes under the same simplified Context", xlabel="x position", ylabel="y position", xlim=(-1.1, 1.1), ylim=(-1.0, 1.0))  # 标注动作空间和显示范围
axis.set_aspect("equal")  # 使用相同比例显示二维几何关系
axis.legend()  # 显示障碍与端点图例
axis.grid(alpha=0.2)  # 添加淡网格帮助观察轨迹间距
fig.tight_layout()  # 调整图像边距
plt.show()  # 显示双峰动作示范

**怎样理解结果：** 蓝色上绕和紫色下绕都从同一起点到达同一目标，并避开灰色障碍。由于简化 Context 没有指定绕行方向，两种轨迹都是正确标签。模型若被要求只输出一条数值轨迹，就必须用某种方式把这两个模式压成一个答案。

## 2. 训练单值回归与双分量混合模型

单值模型直接学习一条 20×2 的平均轨迹并最小化 MSE。混合模型学习两条分量均值、各自尺度和混合权重，使用整条动作块的负对数似然训练。这里省略 Context 网络，只保留输出分布的差别。

In [ ]:
deterministic_chunk = torch.nn.Parameter(torch.zeros(chunk_length, 2))  # 为单值模型建立唯一可学习动作块
deterministic_optimizer = torch.optim.Adam([deterministic_chunk], lr=0.05)  # 使用 Adam 优化平均轨迹
mse_history = []  # 保存单值模型每轮 MSE
for update_index in range(250):  # 重复更新单值轨迹直到收敛
    mse_loss = ((deterministic_chunk - demonstrations) ** 2).mean()  # 对全部示范计算逐位置均方误差
    deterministic_optimizer.zero_grad()  # 清除上一轮单值模型梯度
    mse_loss.backward()  # 计算唯一动作块应调整的方向
    deterministic_optimizer.step()  # 更新单值动作块参数
    mse_history.append(float(mse_loss.detach()))  # 保存当前轮 MSE 供后续绘图
initial_indices = torch.tensor([0, 1])  # 选择两条示范作为混合分量的可重复初始化
mixture_means = torch.nn.Parameter(demonstrations[initial_indices].clone() + 0.1 * torch.randn(2, chunk_length, 2))  # 初始化两条可学习分量均值
mixture_log_scales = torch.nn.Parameter(torch.full((2,), -1.5))  # 用对数参数表示两个分量的正尺度
mixture_logits = torch.nn.Parameter(torch.zeros(2))  # 初始化两个模式具有相同未归一化权重
mixture_optimizer = torch.optim.Adam([mixture_means, mixture_log_scales, mixture_logits], lr=0.03)  # 同时优化均值、尺度和权重
nll_history = []  # 保存混合模型每轮负对数似然
for update_index in range(500):  # 使用更多小步更新连续混合分布
    mixture_scales = mixture_log_scales.exp().clamp(0.02, 0.5)  # 把对数尺度转成稳定的正标准差
    normalized_errors = (demonstrations[:, None] - mixture_means[None]) / mixture_scales[None, :, None, None]  # 计算每条示范相对两个分量的标准化误差
    element_log_probabilities = -0.5 * normalized_errors ** 2 - mixture_log_scales[None, :, None, None] - 0.5 * np.log(2.0 * np.pi)  # 计算各坐标的 Gaussian 对数密度
    chunk_log_probabilities = element_log_probabilities.sum(dim=(2, 3))  # 把一整条动作块的坐标对数密度相加
    weighted_log_probabilities = torch.log_softmax(mixture_logits, dim=0)[None] + chunk_log_probabilities  # 加上两个模式的归一化对数权重
    nll_loss = -torch.logsumexp(weighted_log_probabilities, dim=1).mean()  # 对混合分布求每条示范的负对数似然
    mixture_optimizer.zero_grad()  # 清除上一轮混合模型梯度
    nll_loss.backward()  # 计算分量均值、尺度与权重的梯度
    mixture_optimizer.step()  # 更新完整双分量动作分布
    nll_history.append(float(nll_loss.detach()))  # 保存当前轮负对数似然
fig, axes = plt.subplots(1, 2, figsize=(10, 3.7))  # 创建两种训练目标的收敛曲线
axes[0].plot(mse_history, color="#ea580c")  # 绘制单值动作块的 MSE 变化
axes[0].set(title="Deterministic regression", xlabel="Update", ylabel="MSE")  # 标注单值回归优化目标
axes[1].plot(nll_history, color="#2563eb")  # 绘制混合动作分布的负对数似然变化
axes[1].set(title="Two-component distribution", xlabel="Update", ylabel="Negative log-likelihood")  # 标注混合密度优化目标
for axis in axes:  # 为两幅损失曲线统一添加网格
    axis.grid(alpha=0.2)  # 使用淡网格帮助观察收敛趋势
fig.tight_layout()  # 调整子图间距
plt.show()  # 显示两种 Action Model 的训练过程

**怎样理解结果：** 两条曲线都稳定下降，说明各自训练目标被成功优化。连续密度的 NLL 可以小于零，因为密度值并不是离散概率，不能把它的绝对数值与 MSE 横向比较。真正要检查的是优化后输出位于动作空间的什么位置。

## 3. 检查输出是否仍是一条可执行轨迹

下面把单值输出、两个混合分量和从混合分布抽取的轨迹放回障碍场景。碰撞判定为任意轨迹点同时进入障碍的横纵范围。

In [ ]:
with torch.no_grad():  # 关闭最终结果整理过程的梯度记录
    learned_deterministic = deterministic_chunk.detach().numpy()  # 取出 MSE 学到的唯一动作块
    learned_means = mixture_means.detach().numpy()  # 取出混合模型学到的两条模式均值
    learned_scales = mixture_log_scales.exp().detach().numpy()  # 取出两个动作模式的噪声尺度
    learned_weights = torch.softmax(mixture_logits, dim=0).detach().numpy()  # 取出两个模式的生成概率
sampled_mode_indices = np.random.choice(2, size=30, p=learned_weights)  # 按学习到的混合权重选择三十个动作模式
sampled_chunks = []  # 准备保存从混合分布生成的动作块
for mode_index in sampled_mode_indices:  # 逐个生成具有随机细节的完整轨迹
    sample_noise = np.random.randn(chunk_length, 2) * learned_scales[mode_index]  # 根据该模式尺度生成坐标噪声
    sampled_chunks.append(learned_means[mode_index] + sample_noise)  # 把噪声加到相应模式均值上
sampled_chunks = np.stack(sampled_chunks)  # 组成样本乘时间乘二维坐标数组
def collision_rate(chunks):  # 定义动作块进入中央障碍的比例
    inside_x = np.abs(chunks[:, :, 0]) <= 0.25  # 判断各轨迹点是否位于障碍横向范围
    inside_y = np.abs(chunks[:, :, 1]) <= 0.25  # 判断各轨迹点是否位于障碍纵向范围
    collided = np.any(inside_x & inside_y, axis=1)  # 判断每条轨迹是否至少有一点进入障碍
    return float(collided.mean())  # 返回发生碰撞的动作块比例
deterministic_collision = collision_rate(learned_deterministic[None])  # 计算唯一平均轨迹的碰撞率
mixture_collision = collision_rate(sampled_chunks)  # 计算混合模型样本的碰撞率
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), gridspec_kw={"width_ratios": [2.2, 1.0]})  # 创建轨迹结果与碰撞率对比图
for sampled_chunk in sampled_chunks:  # 绘制混合模型生成的多条动作块
    axes[0].plot(sampled_chunk[:, 0], sampled_chunk[:, 1], color="#93c5fd", alpha=0.25)  # 用浅蓝色显示随机样本
axes[0].plot(learned_deterministic[:, 0], learned_deterministic[:, 1], color="#dc2626", linestyle="--", linewidth=3, label="MSE output")  # 绘制穿过中间的确定性均值
axes[0].plot(learned_means[0, :, 0], learned_means[0, :, 1], color="#2563eb", linewidth=3, label="Mixture component 1")  # 绘制第一个学习模式均值
axes[0].plot(learned_means[1, :, 0], learned_means[1, :, 1], color="#7c3aed", linewidth=3, label="Mixture component 2")  # 绘制第二个学习模式均值
axes[0].add_patch(Rectangle((-0.25, -0.25), 0.5, 0.5, facecolor="#cbd5e1", edgecolor="#475569"))  # 在动作空间中重新绘制障碍
axes[0].set(title="Mean action versus learned modes", xlabel="x position", ylabel="y position", xlim=(-1.1, 1.1), ylim=(-1.0, 1.0))  # 标注轨迹对比含义
axes[0].set_aspect("equal")  # 保持二维动作坐标比例一致
axes[0].legend(loc="upper right", fontsize=8)  # 显示三类主要输出曲线图例
collision_values = [deterministic_collision, mixture_collision]  # 汇总确定性输出与混合样本的碰撞率
bars = axes[1].bar(["MSE mean", "Mixture samples"], collision_values, color=["#dc2626", "#2563eb"])  # 绘制两种模型的碰撞比例
axes[1].set(title="Obstacle collision", ylabel="Rate", ylim=(0.0, 1.1))  # 标注碰撞率范围和含义
for bar, value in zip(bars, collision_values):  # 依次读取两个碰撞率柱子
    axes[1].text(bar.get_x() + bar.get_width() / 2, value + 0.03, f"{value:.2f}", ha="center")  # 在柱子上方写出具体碰撞比例
fig.tight_layout()  # 调整轨迹图与柱状图间距
plt.show()  # 显示单值平均与多峰分布的最终差别

**怎样理解结果：** 红色虚线位于上下模式之间，虽然它同时接近两组示范并取得有限 MSE，却穿过灰色障碍。双分量模型分别保留上绕与下绕，浅蓝样本在两种模式附近变化，碰撞率接近零。混合模型在这里知道有两个模式，但还没有利用 Context 决定何时选择哪一个。

**本练习的结论：** 动作模型的目标不只是减小逐点误差，还要表示完整条件分布并产生可执行样本。混合 Gaussian 需要预先指定分量数；下一章的 Diffusion 与 Flow Matching 将用连续生成过程表示更复杂的高维动作分布。